# Day 1 Lab — Solutions

**Name:** Walaa Omar Hassan  
**Lab:** Build Your First RAG Pipeline (From Scratch!)  
**Exercises completed:** 1 – 7 + Bonus

---


# 🧪 Day 1 Lab: Build Your First RAG Pipeline (From Scratch!)

**Welcome to the lab!** 🎉

Alright, you just survived 3 hours of theory. Now it's time to get your hands dirty.

In this lab, you're going to **build a working RAG pipeline step by step** — from raw text all the way to retrieving relevant chunks using similarity search.

### Rules of the Game:
- Each exercise tells you **what to do** — but **you** write the code.
- Don't overthink it. If you're stuck for more than 5 minutes, ask!
- Google and docs are your friends. This isn't an exam, it's practice.
- Have fun with it. Seriously.

### What You'll Need:
- Python 3.8+
- The packages we'll install below
- Your brain (and maybe some coffee ☕)

---

## 🔧 Setup — Install Everything

Run this cell first. Go grab a coffee while it installs. ☕

In [ ]:
!pip install -q langchain-text-splitters sentence-transformers chromadb numpy scikit-learn tiktoken

---

## 📄 Our Sample Document

Every RAG system starts with documents. Here's ours — a fake (but realistic) university policy document.

**Don't change this cell** — just run it. This is the "knowledge base" your RAG system will search through.

In [ ]:
UNIVERSITY_POLICY = """
# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.

Excused absences include documented medical emergencies, official university activities, and bereavement leave (up to 3 days for immediate family members). Students must submit supporting documentation to the Student Affairs office within 5 business days of the absence.

Instructors are responsible for recording attendance at the beginning of each session. Late arrivals (more than 15 minutes after the scheduled start time) will be recorded as half-absences. Students who arrive more than 30 minutes late will be marked as fully absent.

## Chapter 2: Grading System

The university uses a letter grading system based on the following scale: A+ (95-100%), A (90-94%), B+ (85-89%), B (80-84%), C+ (75-79%), C (70-74%), D+ (65-69%), D (60-64%), F (below 60%). A minimum grade of C is required to pass any course.

Grade Point Average (GPA) is calculated on a 4.0 scale where A+ and A both equal 4.0, B+ equals 3.5, B equals 3.0, C+ equals 2.5, C equals 2.0, D+ equals 1.5, D equals 1.0, and F equals 0.0. Students must maintain a cumulative GPA of 2.0 or higher to remain in good academic standing.

Students who wish to dispute a grade must file a formal Grade Appeal within 14 calendar days of the grade being posted. The appeal must be submitted in writing to the department head, including specific reasons why the student believes the grade is incorrect. The department will form a review committee of three faculty members to evaluate the appeal within 30 days.

## Chapter 3: Examination Rules

Final examinations account for no more than 40% of the total course grade. Midterm examinations account for no more than 25%. The remaining percentage must come from continuous assessment components such as assignments, projects, quizzes, and class participation.

Students must bring a valid university ID card to all examinations. Electronic devices including smartphones, smartwatches, and wireless earbuds are strictly prohibited in the examination hall. Possession of any unauthorized electronic device during an exam will be treated as an academic integrity violation, regardless of whether the device was used.

Make-up examinations are only available for students with documented excused absences. Requests for make-up exams must be submitted within 48 hours of the original exam date. The make-up exam may differ in format and content from the original examination.

## Chapter 4: Academic Integrity

The university maintains a zero-tolerance policy toward plagiarism and academic dishonesty. Plagiarism is defined as presenting someone else's work, ideas, or words as your own without proper attribution. This includes copying from published sources, other students' work, or AI-generated content without proper citation.

First-time offenders will receive a zero on the affected assignment and a formal warning. Second-time offenders will receive an F in the course. Third-time offenders face permanent expulsion from the university. All academic integrity violations are permanently recorded in the student's academic file.

Use of AI tools such as ChatGPT is permitted for research and learning purposes but is strictly prohibited for generating submitted coursework unless the instructor explicitly allows it. Students must disclose any AI assistance used in their work.

## Chapter 5: Student Services and Support

The university provides free tutoring services through the Academic Support Center, located in Building 7, Room 201. Tutoring is available for all core subjects Monday through Thursday from 9:00 AM to 5:00 PM and Fridays from 9:00 AM to 1:00 PM.

Students experiencing mental health difficulties can access free counseling services at the Wellness Center. Appointments can be booked online through the student portal or by calling extension 4455. Emergency walk-in appointments are available during business hours.

The Career Development Office offers resume reviews, mock interviews, and internship placement assistance. Students in their third year and above are eligible for the university's industry partnership program, which guarantees at least one internship interview per semester.

## Chapter 6: Library and Research Facilities

The main library is open from 8:00 AM to 10:00 PM on weekdays and from 10:00 AM to 6:00 PM on weekends. During examination periods, the library extends its hours to midnight on weekdays. Students can borrow up to 10 books at a time with a standard loan period of 14 days. Overdue fines are 2 EGP per day per book, with a maximum fine of 50 EGP per book.

Research databases including IEEE Xplore, SpringerLink, and ScienceDirect are accessible through the university network or VPN. Students can request interlibrary loans for materials not available in the university collection, with a typical processing time of 5-7 business days.

## Chapter 7: Financial Policies

Tuition fees must be paid in full before the start of each semester or through the approved installment plan. The installment plan allows payment in three equal installments due at the beginning, middle, and end of the semester. A late payment fee of 5% will be applied to any overdue installment.

Students who withdraw from a course within the first two weeks of the semester are eligible for a full tuition refund for that course. Withdrawal between weeks 2 and 4 results in a 50% refund. No refund is available after week 4. Scholarship students who withdraw may lose their scholarship for the following semester.
"""

print(f"Document loaded! Length: {len(UNIVERSITY_POLICY)} characters")
print(f"That's roughly {len(UNIVERSITY_POLICY) // 4} tokens (rule of thumb: 1 token ≈ 4 chars)")

---

# Exercise 1: Get to Know Your Document 🔍

Before we do anything fancy, let's actually **look** at what we're working with.

A good RAG engineer always inspects the data first!

### 1.1 — How many chapters does this document have?

Write code to count how many times `"## Chapter"` appears in the text.

*(Yeah, it's simple. But in real life, knowing the structure of your documents is step zero.)*

In [ ]:
# 1.1 — Count the chapters
chapter_count = UNIVERSITY_POLICY.count("## Chapter")
print(f"Number of chapters: {chapter_count}")

# Sanity check with a regex (safer if spacing is inconsistent)
import re
print("Regex check:", len(re.findall(r"^##\s*Chapter", UNIVERSITY_POLICY, flags=re.M)))


### 1.2 — Extract the chapter titles

Get a list of all the chapter titles (the lines that start with `## Chapter`).

Print them nicely.

In [ ]:
# 1.2 — Extract the chapter titles
chapter_titles = [
    line.strip().lstrip("#").strip()
    for line in UNIVERSITY_POLICY.split("\n")
    if line.strip().startswith("## Chapter")
]

print(f"Found {len(chapter_titles)} chapters:\n")
for i, title in enumerate(chapter_titles, start=1):
    print(f"  {i}. {title}")


### 1.3 — Why can't we just paste this entire document into an LLM?

This is a **thinking question** (no code needed). Write your answer in the markdown cell below.

Think about: context window limits, cost, precision, "lost in the middle" problem...

Give at least **3 reasons**.

*Your answer here:*

1. **Context window limits.** Every LLM can only read a limited number of tokens at once. Our policy document is already ~5,900 characters (~1,500 tokens); a real university has hundreds of documents (handbooks, regulations, circulars). They will simply not fit in the prompt, no matter how large the window gets.

2. **Cost and latency.** You pay per token, on *every single* request. Sending the whole document for a question as small as "how many books can I borrow?" means paying for Chapters 1–7 to answer something that lives in one paragraph. It is also slower, because the model has to process everything.

3. **Precision — the "lost in the middle" problem.** Models attend best to the beginning and the end of a long prompt and get noticeably weaker on information buried in the middle. Feeding 7 chapters when only 3 sentences are relevant adds distractors: the model may mix the 75% attendance rule with the 75–79% C+ grade band, because both contain "75%".

4. *(Extra)* **Freshness and maintenance.** Policies change ("Version 3.2"). With retrieval you re-index the changed document; with a giant hard-coded prompt you have to rewrite the prompt, and you have no way of citing *which* chapter an answer came from.


---

# Exercise 2: Text Preprocessing 🧹

Remember from the presentation: preprocessing is **task-dependent**, not a fixed checklist!

Let's see what kind of cleaning makes sense for our university policy document.

### 2.1 — Simulate a "messy" document

Here's a chunk of text that looks like it was badly extracted from a PDF. Your job is to **clean it up**.

Rules:
- Fix the extra whitespace and weird line breaks
- Remove the repeated header/footer junk
- But **keep** the section title and the actual content
- Keep numbers and percentages — they matter!

In [ ]:
messy_text = """
University  Policy    Document — Page 14       CONFIDENTIAL


## Chapter 2:     Grading     System


The    university uses     a letter   grading system     based on
the following    scale:  A+    (95-100%),   A (90-94%),
B+ (85-89%),   B   (80-84%),    C+ (75-79%),
C   (70-74%),   D+    (65-69%),   D   (60-64%),
F   (below    60%).


University  Policy    Document — Page 14       CONFIDENTIAL
"""

# YOUR CODE HERE — clean this text
import re


def clean_pdf_text(raw: str) -> str:
    """Clean text badly extracted from a PDF.

    Strategy:
      1. Drop repeated header/footer lines (page numbers, CONFIDENTIAL stamps).
      2. Split into paragraphs on blank lines (paragraph structure = meaning).
      3. Inside each paragraph, collapse the PDF's hard line-wraps and
         multiple spaces into single spaces.
    Numbers, percentages and the heading are left completely untouched.
    """
    junk = re.compile(r"Page\s*\d+|CONFIDENTIAL", flags=re.IGNORECASE)

    # 1. remove header / footer noise
    lines = [ln for ln in raw.split("\n") if not junk.search(ln)]
    text = "\n".join(lines)

    # 2. + 3. rebuild paragraph by paragraph
    paragraphs = re.split(r"\n\s*\n", text)
    paragraphs = [re.sub(r"\s+", " ", p).strip() for p in paragraphs]
    paragraphs = [p for p in paragraphs if p]  # drop empties

    return "\n\n".join(paragraphs)


cleaned_text = clean_pdf_text(messy_text)

print("=== BEFORE ===")
print(messy_text)
print("\n=== AFTER ===")
print(cleaned_text)

print("\n--- sanity checks ---")
print("Heading kept? ", "## Chapter 2: Grading System" in cleaned_text)
print("Percentages kept?", all(x in cleaned_text for x in ["95-100%", "60-64%", "below 60%"]))
print("Footer gone?    ", "CONFIDENTIAL" not in cleaned_text)
print(f"Size: {len(messy_text)} chars -> {len(cleaned_text)} chars")


### 2.2 — Thinking question: What should you NOT remove?

Look at our university policy document. If you were preprocessing it for a RAG system, which of the following would you **keep** and which would you **remove**? And **why**?

Fill in the table below:

| Item | Keep or Remove? | Why? |
|------|----------------|------|
| Chapter headings ("## Chapter 1: Attendance") | | |
| Percentages like "75%" or "60%" | | |
| The version number "Version 3.2" | | |
| Extra blank lines between paragraphs | | |
| Specific room numbers like "Building 7, Room 201" | | |
| Phone extension "4455" | | |

*Your answer here (fill in the table):*

| Item | Keep or Remove? | Why? |
|------|----------------|------|
| Chapter headings ("## Chapter 1: Attendance") | **Keep** | They carry the topic of the chunk and are the cheapest metadata we will ever get. They let us do structure-aware chunking (Ex 3.6) and metadata filtering (Ex 6.3), and they let the answer cite its source. |
| Percentages like "75%" or "60%" | **Keep** | They *are* the answer. A policy without its numbers ("students must attend a minimum of classes") is worse than useless — it looks confident and is wrong. |
| The version number "Version 3.2" | **Keep** (as metadata, not inside every chunk) | Useful for provenance and for telling an outdated answer from a current one. But it belongs in the document metadata; repeating it in each chunk only adds noise to the embedding. |
| Extra blank lines between paragraphs | **Remove** (collapse them) | They carry no meaning for the embedding model, they waste tokens, and they make chunk-size accounting unreliable. Note: collapse them, but keep *one* separator so paragraph boundaries survive — recursive splitting relies on `\n\n`. |
| Specific room numbers like "Building 7, Room 201" | **Keep** | This is exactly the kind of factual detail students ask about ("where is the tutoring center?"). Stripping "noise-looking" tokens would delete the answer. |
| Phone extension "4455" | **Keep** | Same reason. Short numbers look like junk to a naive cleaner, but this one is actionable information for a student in distress. |

**The general rule:** preprocessing is *task-dependent*. For a policy Q&A system the numbers, headings and locations are the value of the document, so cleaning should only target **formatting artifacts** (wrap-induced line breaks, repeated headers/footers, multiple spaces, page numbers) — never content. Aggressive NLP cleaning that you might use for topic modelling (lowercasing, stop-word removal, stemming, punctuation stripping) would actively *hurt* here: it destroys "A+" vs "A", "2.0" vs "2", and negations like "no refund".


---

# Exercise 3: Chunking — The Fun Part ✂️

Okay this is where it gets interesting. We're going to split our document into chunks using **different strategies** and see how they compare.

Remember: how you chunk your text has a HUGE impact on retrieval quality!

### 3.1 — Fixed-size Character Chunking

Split `UNIVERSITY_POLICY` into chunks of **500 characters** each.

**No overlap** for now. Just chop it up every 500 characters.

Then:
- Print how many chunks you got
- Print the **first 3 chunks** with their character counts
- Look at where the chunks break — do any of them cut a sentence in half?

In [ ]:
# 3.1 — Fixed-size character chunking, no overlap
chunk_size = 500
fixed_chunks = [
    UNIVERSITY_POLICY[i:i + chunk_size]
    for i in range(0, len(UNIVERSITY_POLICY), chunk_size)
]

print(f"Document length : {len(UNIVERSITY_POLICY)} characters")
print(f"Chunk size      : {chunk_size}")
print(f"Number of chunks: {len(fixed_chunks)}\n")

for i, chunk in enumerate(fixed_chunks[:3]):
    print("=" * 70)
    print(f"CHUNK {i}  ({len(chunk)} chars)")
    print("=" * 70)
    print(chunk)
    print()

# Where do the breaks happen?
print("#" * 70)
print("Look at the seam between chunk 0 and chunk 1:")
print("...end of chunk 0 ->", repr(fixed_chunks[0][-60:]))
print("start of chunk 1 ->", repr(fixed_chunks[1][:60]))


### 3.2 — Now add overlap!

Do the same thing but this time with an **overlap of 50 characters**.

- How many chunks do you get now? (should be more than before!)
- Print chunk 2 and chunk 3. Can you see the overlapping text between them?
- Why does overlap help with retrieval? (write a brief answer)

In [ ]:
# 3.2 — Fixed-size chunking WITH overlap
chunk_size = 500
overlap = 50
step = chunk_size - overlap  # 450

overlap_chunks = [
    UNIVERSITY_POLICY[i:i + chunk_size]
    for i in range(0, len(UNIVERSITY_POLICY), step)
]

print(f"Without overlap: {len(fixed_chunks)} chunks")
print(f"With overlap   : {len(overlap_chunks)} chunks  (step = {step})\n")

print("=" * 70)
print("CHUNK 2")
print("=" * 70)
print(overlap_chunks[2])
print()
print("=" * 70)
print("CHUNK 3")
print("=" * 70)
print(overlap_chunks[3])

# Prove the overlap is really there
shared = overlap_chunks[2][-overlap:]
print("\n--- overlapping region ---")
print(repr(shared))
print("Is it the start of chunk 3? ->", overlap_chunks[3].startswith(shared))


### 3.3 — Recursive Chunking with LangChain

Now let's use the **industry standard** — `RecursiveCharacterTextSplitter` from LangChain.

Do the following:
1. Import `RecursiveCharacterTextSplitter` from `langchain_text_splitters`
2. Create a splitter with `chunk_size=500` and `chunk_overlap=50`
3. Split `UNIVERSITY_POLICY` using `.split_text()`
4. Print the number of chunks
5. Print the first 3 chunks

**Compare**: Do these chunks look "smarter" than the fixed-size ones from 3.1? How?

In [ ]:
# 3.3 — Recursive chunking with LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # this is the default order
)

recursive_chunks = splitter.split_text(UNIVERSITY_POLICY)

print(f"Fixed-size chunks (3.1): {len(fixed_chunks)}")
print(f"Recursive chunks  (3.3): {len(recursive_chunks)}\n")

for i, chunk in enumerate(recursive_chunks[:3]):
    print("=" * 70)
    print(f"CHUNK {i}  ({len(chunk)} chars)")
    print("=" * 70)
    print(chunk)
    print()

lengths = [len(c) for c in recursive_chunks]
print(f"min={min(lengths)}  max={max(lengths)}  avg={sum(lengths)/len(lengths):.1f}")


### 3.4 — Experiment with chunk sizes

Let's see how chunk size affects the number of chunks and their quality.

Using `RecursiveCharacterTextSplitter`, split the document with these **3 different chunk sizes**:
- `chunk_size=200` (small)
- `chunk_size=500` (medium)
- `chunk_size=1000` (large)

Keep `chunk_overlap=50` for all.

For each:
1. Print the total number of chunks
2. Print the **average** chunk length (in characters)
3. Pick one chunk from each and read it — which size gives the most "meaningful" chunks?

In [ ]:
# 3.4 — How does chunk size change things?
sizes = [200, 500, 1000]
results = {}

for size in sizes:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    chunks = sp.split_text(UNIVERSITY_POLICY)
    lengths = [len(c) for c in chunks]
    results[size] = chunks
    print(f"chunk_size={size:>4} | chunks={len(chunks):>3} | "
          f"avg={sum(lengths)/len(lengths):>6.1f} | "
          f"min={min(lengths):>4} | max={max(lengths):>4}")

# Read one chunk from each size and judge it by eye
print()
for size in sizes:
    chunks = results[size]
    idx = len(chunks) // 3          # roughly the same position in the document
    print("=" * 70)
    print(f"SAMPLE CHUNK from chunk_size={size}  (chunk #{idx}, {len(chunks[idx])} chars)")
    print("=" * 70)
    print(chunks[idx])
    print()


### 3.5 — Thinking question: Which chunk size would you pick?

For our university policy Q&A system, which chunk size do you think works best? Why?

Think about the tradeoff:
- Too small → precise but loses context
- Too large → more context but less precise, more noise

*Your answer here:*

**My pick: `chunk_size=500` with `chunk_overlap=50`.** (For this specific document I would actually go further and use structure-aware chunking — see 3.6.)

Why, looking at the numbers we just printed:

- **200 is too small.** We get ~48 tiny chunks averaging ~143 characters — often a single sentence, sometimes half of one. A chunk that says *"If attendance drops below 60%, the student will be automatically barred..."* has lost the sentence that told us 60% is about *attendance* and not about a failing grade. Retrieval is precise but the retrieved text can't stand on its own, and the LLM ends up answering from a fragment.

- **1000 is too large.** Only ~7 chunks for the whole document, averaging ~871 characters, so one chunk swallows a whole chapter. Everything gets retrieved for everything; the embedding becomes an average of several topics and loses its sharpness, and we push a lot of irrelevant text into the prompt (cost + "lost in the middle" again).

- **500 is the sweet spot here.** ~19 chunks averaging ~320 characters — roughly one complete paragraph, which in this document is exactly one self-contained rule (the warning threshold, the excused-absence procedure, the appeal deadline). That is both *precise enough* to rank well and *complete enough* to answer from.

The overlap of 50 is cheap insurance: it keeps a rule from being cut exactly at its threshold number.

One honest caveat: the "right" chunk size depends on the *questions*, not only on the document. Our test queries are narrow, factual, single-fact questions ("what is the late fee?"), which favour small chunks. If users asked comparative questions spanning a whole chapter, 1000 would start to look better — and the real answer would be a hybrid: small chunks for retrieval, with the surrounding context expanded before it goes to the LLM.


### 3.6 — Markdown-based / Structure-aware Chunking

Our document has nice Markdown headers (`## Chapter ...`). Let's use them!

Import `MarkdownHeaderTextSplitter` from `langchain_text_splitters` and split the document by its headers.

Use these headers to split on:
```python
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
```

- How many chunks do you get?
- Print each chunk's `metadata` and the first 100 characters of its `page_content`
- How is this different from the recursive chunking approach? Is it better for this document?

In [ ]:
# 3.6 — Structure-aware chunking on Markdown headers
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = md_splitter.split_text(UNIVERSITY_POLICY)

print(f"Recursive chunking gave : {len(recursive_chunks)} chunks")
print(f"Markdown chunking gave  : {len(md_chunks)} chunks\n")

for i, doc in enumerate(md_chunks):
    print("=" * 70)
    print(f"CHUNK {i}  ({len(doc.page_content)} chars)")
    print("metadata:", doc.metadata)
    print("-" * 70)
    print(doc.page_content[:100].replace("\n", " "), "...")
    print()

# Best of both worlds: split by header first, then size-split anything too long.
# This is what you would actually ship.
final_docs = []
size_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
for doc in md_chunks:
    for piece in size_splitter.split_text(doc.page_content):
        final_docs.append({"text": piece, "metadata": dict(doc.metadata)})

print("#" * 70)
print(f"Header-split + size-split = {len(final_docs)} chunks, each carrying its chapter:")
for d in final_docs[:3]:
    print(" -", d["metadata"].get("Header 2"), "|", d["text"][:60].replace("\n", " "), "...")


---

# Exercise 4: Embeddings — Turning Text into Numbers 🔢

Now we're getting to the core of RAG. We need to convert our text chunks into **vectors** (embeddings) so we can search them by **meaning**, not just keywords.

We'll use the `sentence-transformers` library — it's free, local, and works great!

### 4.1 — Load an embedding model

Load the `all-MiniLM-L6-v2` model from `sentence_transformers`.

This is a small, fast model that's perfect for learning (and honestly, good enough for many real applications too).

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
```

After loading it, answer these questions:
- What is the **embedding dimension** of this model? (Hint: encode any sentence and check the `.shape`)
- What does that number mean?

In [ ]:
# 4.1 — Load the embedding model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

test_vector = model.encode("Hello, RAG!")
print("Vector shape:", test_vector.shape)
print("Embedding dimension:", test_vector.shape[0])
print("Max input length (tokens):", model.max_seq_length)


### 4.2 — Embed a sentence and inspect it

Encode this sentence: `"What is the attendance policy?"`

Then:
1. Print the shape of the resulting vector
2. Print the first 10 values of the vector
3. What do these numbers represent? (just a brief answer — no essay needed!)

In [ ]:
# 4.2 — Embed one sentence and look inside it
import numpy as np

sentence = "What is the attendance policy?"
vec = model.encode(sentence)

print("Sentence :", sentence)
print("Shape    :", vec.shape)          # (384,)
print("Dtype    :", vec.dtype)
print("\nFirst 10 values:")
print(vec[:10])
print("\nVector length (L2 norm):", np.linalg.norm(vec))
print("min / max value:", vec.min(), "/", vec.max())


### 4.3 — Semantic similarity experiment 🧪

This is where it gets cool. Let's see if the model actually "understands" meaning!

Encode these 5 sentences:

```python
sentences = [
    "What is the attendance requirement?",        # Query
    "Students must attend 75% of classes.",       # Relevant!
    "The grading scale uses A through F.",        # Different topic
    "Class presence is mandatory for most sessions.",  # Relevant (paraphrase!)
    "The library is open until 10 PM.",           # Totally unrelated
]
```

Then:
1. Compute the **cosine similarity** between the first sentence (the query) and each of the other 4
2. Print the similarity scores
3. Which sentences are most similar to the query? Does this match your intuition?

Hint for cosine similarity:
```python
from sklearn.metrics.pairwise import cosine_similarity
# cosine_similarity(vector_a.reshape(1, -1), vector_b.reshape(1, -1))
```

In [ ]:
# 4.3 — Does the model actually understand meaning?
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sentences = [
    "What is the attendance requirement?",             # 0 - the query
    "Students must attend 75% of classes.",            # 1 - relevant
    "The grading scale uses A through F.",             # 2 - different topic
    "Class presence is mandatory for most sessions.",  # 3 - relevant paraphrase
    "The library is open until 10 PM.",                # 4 - unrelated
]

embeddings = model.encode(sentences)
print("Embeddings matrix shape:", embeddings.shape)   # (5, 384)

query_vec = embeddings[0].reshape(1, -1)
scores = cosine_similarity(query_vec, embeddings[1:])[0]

print(f"\nQuery: {sentences[0]}\n")
ranked = sorted(zip(scores, sentences[1:]), reverse=True)
for rank, (score, sent) in enumerate(ranked, start=1):
    bar = "#" * int(score * 40)
    print(f"{rank}. {score:.4f}  {bar}")
    print(f"   {sent}\n")


### 4.4 — Dense vs Keyword matching

Here's a fun one. Let's see where dense embeddings shine and where they struggle.

Check the similarity between these pairs:

**Pair 1 (Paraphrase — no shared keywords):**
- `"Can I skip class?"`
- `"Students must attend 75% of classes."`

**Pair 2 (Same keywords — different meaning):**
- `"I need to pass the class"`
- `"I need a hall pass for the class"`

**Pair 3 (Exact term match):**
- `"GPA 2.0 requirement"`
- `"cumulative GPA of 2.0 or higher"`

Compute cosine similarity for each pair. What do you notice? When does semantic search work great, and when might keyword search do better?

In [ ]:
# 4.4 — Where dense embeddings shine, and where they struggle
pairs = [
    ("Pair 1 - paraphrase, no shared keywords",
     "Can I skip class?",
     "Students must attend 75% of classes."),
    ("Pair 2 - shared keywords, different meaning",
     "I need to pass the class",
     "I need a hall pass for the class"),
    ("Pair 3 - exact term match",
     "GPA 2.0 requirement",
     "cumulative GPA of 2.0 or higher"),
]

for label, a, b in pairs:
    va = model.encode(a).reshape(1, -1)
    vb = model.encode(b).reshape(1, -1)
    sim = cosine_similarity(va, vb)[0][0]

    # crude keyword baseline: Jaccard overlap of lowercased words
    wa = set(a.lower().replace("?", "").replace(".", "").split())
    wb = set(b.lower().replace("?", "").replace(".", "").split())
    jaccard = len(wa & wb) / len(wa | wb)

    print("=" * 70)
    print(label)
    print(f"  A: {a}")
    print(f"  B: {b}")
    print(f"  cosine similarity : {sim:.4f}")
    print(f"  keyword overlap   : {jaccard:.4f}   (shared words: {sorted(wa & wb)})")
    print()


### 4.5 — Embed ALL the chunks!

Now let's embed our actual document chunks.

1. Use the `RecursiveCharacterTextSplitter` from Exercise 3.3 (chunk_size=500, overlap=50) to get chunks
2. Embed ALL chunks using `model.encode(chunks)`
3. Print the shape of the resulting embedding matrix
4. What does each dimension of this matrix represent? (rows = ?, columns = ?)

In [ ]:
# 4.5 — Embed every chunk of the document
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(UNIVERSITY_POLICY)

chunk_embeddings = model.encode(chunks, show_progress_bar=True)

print(f"\nNumber of chunks     : {len(chunks)}")
print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"  rows    = {chunk_embeddings.shape[0]}  -> one row per chunk")
print(f"  columns = {chunk_embeddings.shape[1]}  -> the 384 dimensions of the model")
print(f"\nMemory used: {chunk_embeddings.nbytes / 1024:.1f} KB")


---

# Exercise 5: Similarity Search — Finding the Right Chunks 🎯

Now that we have chunks and their embeddings, let's do what RAG is all about: **retrieval**!

Given a user question, find the most relevant chunks.

### 5.1 — Manual similarity search

Let's do this **from scratch** first — no fancy libraries. Just numpy and cosine similarity.

Given this query: `"What happens if I miss too many classes?"`

1. Embed the query using the same model
2. Compute the cosine similarity between the query embedding and ALL chunk embeddings
3. Sort the results by similarity score (highest first)
4. Print the **top 3** most relevant chunks with their similarity scores

Does the result make sense? Is the system retrieving chunks about attendance?

In [ ]:
# 5.1 — Similarity search from scratch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

query = "What happens if I miss too many classes?"

# Step 1: embed the query with the SAME model used for the chunks
query_embedding = model.encode(query).reshape(1, -1)

# Step 2: cosine similarity against every chunk at once
similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

# Step 3: sort, highest first
top_k = 3
top_indices = np.argsort(similarities)[::-1][:top_k]

# Step 4: show the results
print(f"QUERY: {query}\n")
for rank, idx in enumerate(top_indices, start=1):
    print("=" * 70)
    print(f"RANK {rank} | chunk #{idx} | similarity = {similarities[idx]:.4f}")
    print("=" * 70)
    print(chunks[idx])
    print()

print(f"(worst chunk scored {similarities.min():.4f} — the gap is what makes ranking work)")


### 5.2 — Try different queries!

Test your retrieval system with these queries and print the **top 2** chunks for each:

1. `"How is GPA calculated?"`
2. `"Can I use ChatGPT for my assignments?"`
3. `"Where can I get mental health support?"`
4. `"What is the late payment fee?"`

For each query, check: does the top result actually answer the question? 👀

**Bonus**: Try writing a query that the system gets WRONG. Can you break it?

In [ ]:
# 5.2 — A reusable search function
def search(query, chunks, chunk_embeddings, model, top_k=2):
    """Return the top_k (index, score, text) tuples for a query."""
    q = model.encode(query).reshape(1, -1)
    sims = cosine_similarity(q, chunk_embeddings)[0]
    idxs = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i]), chunks[i]) for i in idxs]


queries = [
    "How is GPA calculated?",
    "Can I use ChatGPT for my assignments?",
    "Where can I get mental health support?",
    "What is the late payment fee?",
]

for q in queries:
    print("#" * 70)
    print(f"QUERY: {q}")
    print("#" * 70)
    for rank, (idx, score, text) in enumerate(search(q, chunks, chunk_embeddings, model, top_k=2), 1):
        print(f"\n  [{rank}] chunk #{idx}  score={score:.4f}")
        print("  " + text.replace("\n", "\n  ")[:400])
    print("\n")


# BONUS — trying to break it
print("*" * 70)
print("BONUS: queries designed to fail")
print("*" * 70)

breaking_queries = [
    "How much do I pay if I return a book two weeks late?",   # needs 2 EGP/day AND the 50 EGP cap AND arithmetic
    "What is 75%?",                                           # ambiguous: attendance 75% or the C+ band 75-79%?
    "Is there a dress code?",                                 # not in the document at all
]

for q in breaking_queries:
    idx, score, text = search(q, chunks, chunk_embeddings, model, top_k=1)[0]
    print(f"\nQUERY: {q}")
    print(f"  top score = {score:.4f} (chunk #{idx})")
    print("  " + text.replace("\n", " ")[:220] + " ...")

print("\nNotice: the dress-code question still returns a chunk with a plausible-looking")
print("score. Embedding search ALWAYS returns something — it has no concept of")
print("'not in the document'. That is why a similarity threshold and a")
print("'answer only from the context' instruction both matter.")


### 5.3 — Compare similarity metrics

Let's compare the three metrics we learned about.

For the query `"What is the grading system?"`, compute:
1. **Cosine Similarity** (using `sklearn.metrics.pairwise.cosine_similarity`)
2. **Dot Product** (using `numpy.dot`)
3. **Euclidean Distance** (using `numpy.linalg.norm(a - b)` or `sklearn.metrics.pairwise.euclidean_distances`)

Rank the top 3 chunks using each metric. Do they give the **same ranking** or different ones?

Remember:
- Cosine & Dot Product: **Higher** = more similar
- Euclidean Distance: **Lower** = more similar

In [ ]:
# 5.3 — Cosine vs dot product vs Euclidean distance
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

query = "What is the grading system?"
q_vec = model.encode(query).reshape(1, -1)

cos_scores = cosine_similarity(q_vec, chunk_embeddings)[0]
dot_scores = np.dot(chunk_embeddings, q_vec.T).flatten()
euc_dists  = euclidean_distances(q_vec, chunk_embeddings)[0]

top_cos = np.argsort(cos_scores)[::-1][:3]   # higher = better
top_dot = np.argsort(dot_scores)[::-1][:3]   # higher = better
top_euc = np.argsort(euc_dists)[:3]          # LOWER  = better

print(f"QUERY: {query}\n")
print(f"{'Rank':<6}{'Cosine':<22}{'Dot product':<22}{'Euclidean':<22}")
print("-" * 72)
for r in range(3):
    print(f"{r+1:<6}"
          f"#{top_cos[r]:<3} ({cos_scores[top_cos[r]]:.4f})     "
          f"#{top_dot[r]:<3} ({dot_scores[top_dot[r]]:.4f})     "
          f"#{top_euc[r]:<3} ({euc_dists[top_euc[r]]:.4f})")

print("\nSame ranking from all three?",
      list(top_cos) == list(top_dot) == list(top_euc))

# Why? Check whether the model already returns unit-length vectors.
norms = np.linalg.norm(chunk_embeddings, axis=1)
print(f"\nChunk vector norms: min={norms.min():.4f}, max={norms.max():.4f}")
print("If every norm is ~1.0 the vectors are normalised, and then:")
print("  dot(a,b) == cosine(a,b)   and   euclidean^2 == 2 - 2*cosine")
print("so the three metrics are guaranteed to produce the SAME ordering.")
print("On un-normalised vectors they would differ: dot product rewards long")
print("vectors (it is biased toward longer text), cosine only cares about direction.")

print("\nWinning chunk:\n")
print(chunks[top_cos[0]])


---

# Exercise 6: Vector Database with ChromaDB 🗄️

Doing similarity search manually with numpy is fine for learning, but in practice we use a **vector database**.

Let's use **ChromaDB** — it's easy to set up and perfect for this lab.

### 6.1 — Create a Chroma collection and add your chunks

1. Import `chromadb` and create an in-memory client
2. Create a collection called `"university_policy"`
3. Add all your chunks to the collection
   - Each chunk needs a unique `id` (e.g., `"chunk_0"`, `"chunk_1"`, ...)
   - Add the chunk text as `documents`
   - Add some metadata for each chunk — at minimum, add `{"chunk_index": i}` for each

Here's a skeleton to get you started:
```python
import chromadb
client = chromadb.Client()  # in-memory
collection = client.create_collection(name="university_policy")

# Add documents...
```

After adding, print: `collection.count()` to verify everything got in.

In [ ]:
# 6.1 — Put the chunks in ChromaDB
import chromadb

client = chromadb.Client()  # in-memory, resets when the runtime restarts

# start clean if the cell is re-run
try:
    client.delete_collection("university_policy")
except Exception:
    pass

collection = client.create_collection(
    name="university_policy",
    metadata={"hnsw:space": "cosine"},  # use cosine distance instead of the L2 default
)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=[{"chunk_index": i, "char_length": len(c)} for i, c in enumerate(chunks)],
)

print("Documents in collection:", collection.count())

peek = collection.peek(limit=2)
print("\nSample ids       :", peek["ids"])
print("Sample metadatas :", peek["metadatas"])
print("\nNote: we did not pass embeddings, so Chroma embedded the text itself")
print("with its default model (all-MiniLM-L6-v2 — the same one we used).")


### 6.2 — Query the vector database

Now query your collection!

Use `collection.query()` with the following queries and `n_results=3`:

1. `"What is the penalty for cheating?"`
2. `"How do I get a tutor?"`
3. `"Can I get a refund if I drop a course?"`

For each result, print:
- The returned documents
- The distances (similarity scores)

Does ChromaDB give you the same results as your manual search from Exercise 5? 🤔

In [ ]:
# 6.2 — Query the vector database
db_queries = [
    "What is the penalty for cheating?",
    "How do I get a tutor?",
    "Can I get a refund if I drop a course?",
]

for q in db_queries:
    results = collection.query(query_texts=[q], n_results=3)

    print("#" * 70)
    print(f"QUERY: {q}")
    print("#" * 70)
    for rank, (doc, dist, meta) in enumerate(
        zip(results["documents"][0], results["distances"][0], results["metadatas"][0]), 1
    ):
        # with cosine space: distance = 1 - cosine_similarity
        print(f"\n  [{rank}] chunk #{meta['chunk_index']} | distance={dist:.4f} | similarity≈{1 - dist:.4f}")
        print("  " + doc.replace("\n", " ")[:300] + " ...")
    print("\n")


# Same results as the manual numpy search from Exercise 5?
print("*" * 70)
print("COMPARISON: manual numpy search vs ChromaDB")
print("*" * 70)
for q in db_queries:
    manual = [i for i, _, _ in search(q, chunks, chunk_embeddings, model, top_k=3)]
    chroma = [m["chunk_index"] for m in collection.query(query_texts=[q], n_results=3)["metadatas"][0]]
    print(f"\n{q}")
    print(f"  manual : {manual}")
    print(f"  chroma : {chroma}")
    print(f"  match  : {manual == chroma}")

print("\nThey should agree (same model, same metric). Chroma uses an approximate")
print("nearest-neighbour index (HNSW), so on a big corpus small differences are")
print("normal and expected — that is the speed/accuracy trade-off you buy.")


### 6.3 — Filtering with metadata!

One of the superpowers of vector databases is **metadata filtering**.

Let's make this more useful. Delete the old collection and create a new one where each chunk has richer metadata.

For each chunk, figure out which chapter it belongs to and add `{"chapter": "Chapter X: ...."}` as metadata.

Hint: You could check if the chunk contains certain keywords or use the chapter titles you extracted in Exercise 1.2. Don't overthink it — a simple approach is fine!

Then query with a **metadata filter**:
```python
collection.query(
    query_texts=["What is the late fee?"],
    n_results=3,
    where={"chapter": "Chapter 7: Financial Policies"}
)
```

Compare the results with and without the filter. Does filtering help?

In [ ]:
# 6.3 — Richer metadata + filtered queries
import re

# --- figure out which chapter each chunk belongs to -------------------------
# Walk the document once and record where each chapter starts, then map every
# chunk to the last chapter heading that appeared before it.
chapter_positions = [
    (m.start(), m.group(0).lstrip("#").strip())
    for m in re.finditer(r"^##\s*Chapter.*$", UNIVERSITY_POLICY, flags=re.M)
]


def chapter_for(chunk_text: str) -> str:
    pos = UNIVERSITY_POLICY.find(chunk_text[:80])   # locate the chunk in the doc
    if pos == -1:
        return "Unknown"
    current = "Front matter"
    for start, title in chapter_positions:
        if start <= pos:
            current = title
        else:
            break
    return current


chunk_metadatas = [
    {"chunk_index": i, "chapter": chapter_for(c), "char_length": len(c)}
    for i, c in enumerate(chunks)
]

from collections import Counter
print("Chunks per chapter:")
for chapter, n in Counter(m["chapter"] for m in chunk_metadatas).items():
    print(f"  {n:>2}  {chapter}")

# --- rebuild the collection with the richer metadata ------------------------
try:
    client.delete_collection("university_policy")
except Exception:
    pass

collection = client.create_collection(
    name="university_policy",
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=chunk_metadatas,
)
print("\nRe-indexed:", collection.count(), "chunks\n")

# --- same query, with and without a filter ----------------------------------
q = "What is the late fee?"

print("=" * 70)
print("WITHOUT filter")
print("=" * 70)
res = collection.query(query_texts=[q], n_results=3)
for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
    print(f"  [{meta['chapter']}] dist={dist:.4f}")
    print("   " + doc.replace("\n", " ")[:160] + " ...\n")

print("=" * 70)
print('WITH filter: chapter = "Chapter 7: Financial Policies"')
print("=" * 70)
res = collection.query(
    query_texts=[q],
    n_results=3,
    where={"chapter": "Chapter 7: Financial Policies"},
)
for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
    print(f"  [{meta['chapter']}] dist={dist:.4f}")
    print("   " + doc.replace("\n", " ")[:160] + " ...\n")

print("Why this matters: 'late fee' is ambiguous in our document — there is a 5%")
print("late PAYMENT fee (Chapter 7) and a 2 EGP/day overdue LIBRARY fine")
print("(Chapter 6). Unfiltered search happily mixes them. The filter turns a")
print("fuzzy semantic guess into a scoped, reliable lookup.")
print("\nThe catch: someone has to KNOW to apply the filter. In a real system a")
print("small router/classifier maps the user question to a chapter first — that")
print("is query routing, and it is Day 2 material.")


---

# Exercise 7: Put It All Together — Mini RAG Pipeline! 🚀

Alright, final boss. Let's combine everything into one clean pipeline.

You're NOT going to call an actual LLM (we'll save that for Day 2), but you'll build **everything else**: the indexing pipeline and the retrieval pipeline.

### 7.1 — Build the complete indexing pipeline

Write a function called `build_index` that takes a raw document string and:

1. **Preprocesses** it (basic cleaning — remove extra whitespace, etc.)
2. **Chunks** it using RecursiveCharacterTextSplitter (pick your favorite chunk size)
3. **Stores** the chunks in a ChromaDB collection with metadata
4. Returns the collection

```python
def build_index(document: str, collection_name: str = "rag_index") -> chromadb.Collection:
    # YOUR CODE HERE
    pass
```

In [ ]:
# 7.1 — The full indexing pipeline
import re
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter


def preprocess(document: str) -> str:
    """Formatting-only cleaning. Never touches numbers, headings or content."""
    junk = re.compile(r"Page\s*\d+\s*$|CONFIDENTIAL", flags=re.IGNORECASE)
    lines = [ln for ln in document.split("\n") if not junk.search(ln)]
    text = "\n".join(lines)
    text = re.sub(r"[ \t]+", " ", text)        # collapse runs of spaces
    text = re.sub(r"\n{3,}", "\n\n", text)     # at most one blank line
    return text.strip()


def detect_chapter(chunk_text: str, source: str) -> str:
    """Map a chunk back to the chapter heading it falls under."""
    positions = [
        (m.start(), m.group(0).lstrip("#").strip())
        for m in re.finditer(r"^##\s*Chapter.*$", source, flags=re.M)
    ]
    pos = source.find(chunk_text[:80])
    if pos == -1:
        return "Unknown"
    current = "Front matter"
    for start, title in positions:
        if start <= pos:
            current = title
        else:
            break
    return current


def build_index(document, collection_name="rag_index", chunk_size=500, chunk_overlap=50):
    """Build a searchable index from a raw document.

    preprocess -> chunk -> store in ChromaDB with metadata -> return collection
    """
    # 1. preprocess
    clean = preprocess(document)

    # 2. chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    doc_chunks = splitter.split_text(clean)

    # 3. store
    client = chromadb.Client()
    try:
        client.delete_collection(collection_name)   # idempotent re-runs
    except Exception:
        pass

    coll = client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )
    coll.add(
        ids=[f"chunk_{i}" for i in range(len(doc_chunks))],
        documents=doc_chunks,
        metadatas=[
            {
                "chunk_index": i,
                "chapter": detect_chapter(c, clean),
                "char_length": len(c),
            }
            for i, c in enumerate(doc_chunks)
        ],
    )

    # 4. return
    return coll


# Test it:
collection = build_index(UNIVERSITY_POLICY)
print(f"Index built! Total chunks: {collection.count()}")
print("Sample metadata:", collection.peek(limit=3)["metadatas"])


### 7.2 — Build the retrieval function

Write a function called `retrieve` that:

1. Takes a user query and a ChromaDB collection
2. Searches for the top-K most relevant chunks
3. Returns the chunks formatted nicely as a **context string** that you could paste into an LLM prompt

```python
def retrieve(query: str, collection, top_k: int = 3) -> str:
    # YOUR CODE HERE
    pass
```

The output should look something like:
```
[Source 1] (similarity: 0.87)
Students must attend 75% of classes...

[Source 2] (similarity: 0.72)
Excused absences include...
```

In [ ]:
# 7.2 — The retrieval function
def retrieve(query, collection, top_k=3, min_similarity=0.0):
    """Retrieve the most relevant chunks for a query, formatted as LLM context."""
    results = collection.query(query_texts=[query], n_results=top_k)

    docs = results["documents"][0]
    dists = results["distances"][0]
    metas = results["metadatas"][0]

    blocks = []
    for rank, (doc, dist, meta) in enumerate(zip(docs, dists, metas), start=1):
        similarity = 1 - dist          # cosine space
        if similarity < min_similarity:
            continue
        chapter = meta.get("chapter", "Unknown")
        blocks.append(
            f"[Source {rank}] (similarity: {similarity:.2f} | {chapter})\n"
            f"{doc.strip()}"
        )

    if not blocks:
        return "NO_RELEVANT_CONTEXT_FOUND"

    return "\n\n".join(blocks)


# Test it:
result = retrieve("What happens if I cheat on an exam?", collection)
print(result)


### 7.3 — Build a fake "answer" function (prompt template)

We don't have an LLM today, but let's prepare the **prompt** that we WOULD send to one.

Write a function `build_prompt` that:
1. Takes a user query and the retrieved context (from your `retrieve` function)
2. Returns a nicely formatted prompt string

Use this template (or make your own!):

```
You are a helpful university assistant. Answer the student's question based ONLY on the provided context. If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{retrieved_context}

Question: {query}

Answer:
```

Test it with a few queries and print the full prompts. Imagine you're the LLM — could YOU answer the question from the given context?

In [ ]:
# 7.3 — The prompt template
PROMPT_TEMPLATE = """You are a helpful university assistant. Answer the student's \
question based ONLY on the provided context. If the answer is not in the context, \
say "I don't have enough information to answer that." Quote the exact numbers, \
deadlines and percentages from the context, and mention which chapter they come from.

Context:
{retrieved_context}

Question: {query}

Answer:"""


def build_prompt(query, context):
    """Build a prompt for the LLM."""
    return PROMPT_TEMPLATE.format(retrieved_context=context, query=query)


# Test the full pipeline:
query = "Can I use AI tools for my homework?"
context = retrieve(query, collection)
prompt = build_prompt(query, context)
print(prompt)

print("\n" + "=" * 70)
print(f"Prompt length: {len(prompt)} characters (~{len(prompt)//4} tokens)")
print(f"Whole document: {len(UNIVERSITY_POLICY)} characters (~{len(UNIVERSITY_POLICY)//4} tokens)")
print(f"Saving: {100 - 100*len(prompt)/len(UNIVERSITY_POLICY):.0f}% fewer tokens per question.")
print("That is the entire point of RAG, in one number.")


### 7.4 — Stress test your pipeline! 🏋️

Run these 6 queries through your complete pipeline (build_index → retrieve → build_prompt).

For each query, check:
- ✅ Did it retrieve the right context?
- ❌ Did it retrieve something irrelevant?
- 🤔 Could an LLM answer correctly from this context?

```python
test_queries = [
    "What percentage of classes do I need to attend?",
    "What grade do I need to pass a course?",
    "Where is the tutoring center located?",
    "What happens if I plagiarize for the second time?",
    "How many books can I borrow from the library?",
    "What is the refund policy if I drop a course after week 3?",
]
```

Write your observations below!

In [ ]:
# 7.4 — Stress test
test_queries = [
    "What percentage of classes do I need to attend?",
    "What grade do I need to pass a course?",
    "Where is the tutoring center located?",
    "What happens if I plagiarize for the second time?",
    "How many books can I borrow from the library?",
    "What is the refund policy if I drop a course after week 3?",
]

# The fact each query is really asking about — used to auto-check the retrieval
expected_evidence = [
    "75%",
    "minimum grade of C",
    "Building 7, Room 201",
    "Second-time offenders",
    "10 books",
    "50% refund",
]

for i, (q, evidence) in enumerate(zip(test_queries, expected_evidence), start=1):
    context = retrieve(q, collection, top_k=3)
    prompt = build_prompt(q, context)
    found = evidence.lower() in context.lower()

    print("#" * 72)
    print(f"QUERY {i}: {q}")
    print(f"expected evidence: '{evidence}'  ->  {'FOUND' if found else 'MISSING'}")
    print("#" * 72)
    print(context[:900])
    print("\n")


*Your observations:*

- Query 1: "What percentage of classes do I need to attend?" — **✅** — Top chunk is the opening paragraph of Chapter 1 with the 75% minimum, the 15% warning threshold and the 60% exam bar. The wording of the question is almost the wording of the document, so this is the easy case.
- Query 2: "What grade do I need to pass a course?" — **✅** — Retrieves the Chapter 2 grading-scale paragraph containing "A minimum grade of C is required to pass any course." Worth noticing that the chunk *also* contains the full A+/A/B+ scale, so the LLM has to pick the right sentence out of a paragraph full of distracting numbers.
- Query 3: "Where is the tutoring center located?" — **✅** — Chapter 5, with "Building 7, Room 201" and the opening hours. A good example of why we refused to strip "junk-looking" details in Exercise 2.2.
- Query 4: "What happens if I plagiarize for the second time?" — **✅ (with a caveat)** — The penalty-ladder chunk comes back and contains "Second-time offenders will receive an F in the course." The risk here is *ordinal* confusion: first/second/third offences all live in the same chunk, and a weaker model can grab the wrong line. Retrieval did its job; correctness now depends on the generation step.
- Query 5: "How many books can I borrow from the library?" — **✅** — Chapter 6, "up to 10 books... loan period of 14 days". Note that "14 days" (loan period) and "14 calendar days" (grade appeal, Chapter 2) are different facts that share a number — a reminder that numbers alone are not identifiers.
- Query 6: "What is the refund policy if I drop a course after week 3?" — **⚠️ partially** — The right chunk (Chapter 7 withdrawal paragraph) is retrieved, but the document never says "week 3": it says "between weeks 2 and 4 results in a 50% refund." The system retrieves correctly and the answer *is* derivable, but only if the model performs the reasoning step "week 3 falls inside the 2–4 window." Retrieval is not the bottleneck here — reasoning is.

**Overall:** 5 clean hits and 1 that needs inference. Retrieval quality is high because the document is short, well structured, and the questions use vocabulary close to the source. The weak spots are all in the *last mile*: chunks that bundle several similar numbers together, and questions that require arithmetic or range-checking rather than lookup.


---

# Bonus Exercise: Break Your Own System! 💥

*(Optional but highly recommended)*

Every RAG system has weaknesses. Let's find yours!

Try to come up with queries that your system handles **badly**. Things like:

- Questions that need information from **multiple chapters** to answer
- Very **vague** questions
- Questions about things that are **NOT** in the document
- Questions with **different wording** than what's in the document
- Questions about **numbers or specific values**

For each failing query, think about: **What would fix this?** (hint: better chunking? bigger chunks? hybrid search? metadata filtering?)

This is actually one of the most important skills in RAG engineering — knowing where your system fails and how to improve it! 🧠

In [ ]:
# Bonus — break the system on purpose
tricky_queries = [
    # 1. multi-hop: the answer lives in TWO different chapters
    "If I'm barred from the final exam, does that affect my GPA and my scholarship?",

    # 2. arithmetic / not stated literally
    "How much do I owe if I return a book 40 days late?",

    # 3. NOT in the document at all
    "What is the university's dress code?",

    # 4. vague
    "What are the rules?",

    # 5. ambiguous number — 'the 75% rule' means two different things
    "Tell me about the 75% rule",

    # 6. negation — the document states the absence of something
    "When can I get a refund after week 5?",

    # 7. completely different vocabulary from the document
    "Can I bring my Apple Watch to the test?",
]

for i, q in enumerate(tricky_queries, start=1):
    results = collection.query(query_texts=[q], n_results=2)
    print("#" * 72)
    print(f"TRICKY {i}: {q}")
    print("#" * 72)
    for doc, dist, meta in zip(results["documents"][0], results["distances"][0], results["metadatas"][0]):
        print(f"  [{meta['chapter']}] similarity≈{1 - dist:.3f}")
        print("   " + doc.replace("\n", " ")[:200] + " ...\n")


# The most dangerous failure mode, isolated:
print("*" * 72)
print("THE SILENT FAILURE")
print("*" * 72)
out_of_scope = "What is the university's dress code?"
in_scope = "What is the attendance requirement?"
for q in (in_scope, out_of_scope):
    d = collection.query(query_texts=[q], n_results=1)["distances"][0][0]
    print(f"  similarity≈{1 - d:.3f}   {q}")
print("\nThe out-of-scope question does NOT score zero. Vector search always")
print("returns its nearest neighbour, however far away it is. Without a")
print("similarity threshold the pipeline confidently hands the LLM irrelevant")
print("context — and that is where hallucinations are born.")


*What queries broke the system and why?*

1. **Multi-hop: "If I'm barred from the final exam, does that affect my GPA and my scholarship?"**
   The answer is spread across three chapters — attendance barring (Ch. 1), the GPA 2.0 good-standing rule (Ch. 2) and the scholarship clause (Ch. 7). Top-k retrieval on a single embedding returns chunks that are all near *one* of those, so at least one hop is always missing.
   **Fix:** query decomposition (split the question into sub-questions and retrieve for each), or a larger k combined with a reranker that keeps the genuinely relevant ones.

2. **Arithmetic: "How much do I owe if I return a book 40 days late?"**
   The correct chunk *is* retrieved (2 EGP/day, 50 EGP maximum) but the answer — 50 EGP, because 80 exceeds the cap — is never written anywhere. Retrieval cannot fail *or* succeed here; the gap is reasoning.
   **Fix:** not a retrieval fix. A stronger model, chain-of-thought prompting, or a calculator tool.

3. **Out of scope: "What is the university's dress code?"**
   The most dangerous one, because it fails *silently*. The nearest chunk still comes back with a respectable-looking similarity, and a compliant LLM may invent a dress code from it.
   **Fix:** a minimum-similarity threshold (drop anything below ~0.3), plus the explicit "if the answer is not in the context, say you don't know" instruction we put in the prompt template — and ideally a groundedness check on the generated answer.

4. **Ambiguity: "Tell me about the 75% rule"**
   75% appears as the attendance minimum (Ch. 1) and as the bottom of the C+ band (Ch. 2). Dense embeddings match on topic, not on which "75%" the user meant.
   **Fix:** metadata filtering / query routing, or an interactive clarifying question.

5. **Vagueness: "What are the rules?"**
   The query embedding sits in the middle of the whole document, so the top-k is essentially arbitrary and unstable.
   **Fix:** query rewriting before retrieval, or detecting low score *spread* (when the top 5 results all score nearly the same, the query is under-specified) and asking the user to narrow it.

6. **Vocabulary mismatch: "Can I bring my Apple Watch to the test?"**
   The document says "smartwatches" and "examinations", never "Apple Watch" or "test". This is where dense retrieval genuinely earns its keep and usually succeeds — but if it had said "my Garmin Fenix 7", a pure keyword system would score zero while the embedding still lands close.
   **Fix:** hybrid search (BM25 + dense, fused with RRF) covers both directions — rare exact tokens *and* paraphrases.

**The pattern:** every failure above is one of four things — the answer is *split* (multi-hop), *derived* (reasoning), *absent* (out of scope), or *ambiguous* (needs routing). Better chunking only helps the first one. Knowing which of the four you are looking at is the actual skill.


---

# 🎉 Lab Complete!

Congrats! You just built a RAG pipeline from scratch. Not bad for Day 1, right?

### What you accomplished today:

- ✅ Explored a real document and understood its structure
- ✅ Preprocessed text (and learned when NOT to clean too aggressively)
- ✅ Chunked documents using multiple strategies and compared them
- ✅ Created embeddings and verified they capture semantic meaning
- ✅ Built similarity search from scratch with numpy
- ✅ Used a vector database (ChromaDB) with metadata filtering
- ✅ Assembled a complete retrieval pipeline
- ✅ Stress-tested your system and found its weaknesses

### Coming up in Day 2:

Tomorrow we'll add the **intelligence layer**: reranking, query transformation, prompt engineering, hallucination handling, and **actually connecting an LLM** to get real answers.

We'll also learn how to **evaluate** whether your RAG system is actually good or not.

See you tomorrow! 🚀